In [ ]:
argumfrom selenium import webdriver
from selenium.webdriver.common.by import By
import time
import os
import shutil
import pandas as pd
import requests

# Load TDR list
df = pd.read_excel("MP_pipe_data.xlsx")
tdr_list = df["TDR"].astype(str).tolist()

# Setup download directory
base_download_dir = os.path.join(os.getcwd(), "BOQ_Downloads")
os.makedirs(base_download_dir, exist_ok=True)

# Chrome options
options = webdriver.ChromeOptions()
prefs = {
    "download.default_directory": base_download_dir,
    "download.prompt_for_download": False,
    "safebrowsing.enabled": True
}
options.add_experimental_option("prefs", prefs)
driver = webdriver.Chrome(options=options)

# Manual login
driver.get("https://www.tenderdetail.com/Account/LogOn")
input("🔐 Log in manually, then press ENTER to continue...")

summary = []

# Loop through each TDR
for tdr in tdr_list:
    print(f"\n🔎 Searching for TDR: {tdr}")
    tdr_dir = os.path.join(base_download_dir, tdr)
    os.makedirs(tdr_dir, exist_ok=True)

    # Skip already processed TDR
    existing_files = os.listdir(tdr_dir)
    has_boq = any(f.lower().endswith(".xls") for f in existing_files)
    has_nit = any("nit" in f.lower() and f.lower().endswith(".pdf") for f in existing_files)
    if has_boq and has_nit:
        print(f"⏭️ Skipping {tdr}: already downloaded.")
        summary.append({"TDR": tdr, "BOQ Files": "Already Present", "NIT Files": "Already Present"})
        continue

    # Load tender detail page
    try:
        driver.get(f"https://www.tenderdetail.com/registeruser/searchtenders?ServiceType=1&nm={tdr}")
        time.sleep(2)
    except Exception as e:
        print(f"❌ Connection error: {e}")
        summary.append({"TDR": tdr, "BOQ Files": "ERROR", "NIT Files": "ERROR"})
        continue

    # Validate TDR page
    try:
        tender_brief = driver.find_element(By.CSS_SELECTOR, "#tenderbreif > span").text
        if tdr not in tender_brief:
            summary.append({"TDR": tdr, "BOQ Files": "", "NIT Files": ""})
            continue
    except:
        summary.append({"TDR": tdr, "BOQ Files": "", "NIT Files": ""})
        continue

    # Click view notice
    try:
        view_notice = driver.find_element(By.CSS_SELECTOR, ".viewnotice")
        driver.execute_script("ents[0].click();", view_notice)
        time.sleep(2)
    except:
        summary.append({"TDR": tdr, "BOQ Files": "", "NIT Files": ""})
        continue

    driver.switch_to.window(driver.window_handles[-1])
    time.sleep(2)

    boq_files = []
    nit_files = []

    # Extract all download links
    try:
        all_links = driver.find_elements(By.TAG_NAME, "a")
        for link in all_links:
            href = link.get_attribute("href")
            if not href or "FileName=" not in href:
                continue

            filename = href.split("FileName=")[-1]

            # BOQ XLS via Selenium
            if filename.lower().endswith(".xls") and "boq" in filename.lower():
                print(f"📥 BOQ via Selenium: {filename}")
                driver.get(href)
                boq_files.append(filename)
                time.sleep(2)

            # NIT PDF via Requests
            elif filename.lower().endswith(".pdf") and "nit" in filename.lower():
                print(f"📥 NIT via Requests: {filename}")
                try:
                    response = requests.get(href, timeout=10)
                    if response.ok:
                        with open(os.path.join(tdr_dir, filename), "wb") as f:
                            f.write(response.content)
                        nit_files.append(filename)
                    else:
                        print(f"⚠️ Failed to download: {filename}")
                except Exception as e:
                    print(f"❌ Request error: {e}")
    except Exception as e:
        print(f"⚠️ Link parsing error for {tdr}: {e}")

    # Move downloaded BOQ from base dir to TDR dir
    try:
        for f in os.listdir(base_download_dir):
            if f.lower().endswith(".xls") or f.lower().endswith(".xlsx"):
                src = os.path.join(base_download_dir, f)
                dst = os.path.join(tdr_dir, f)
                shutil.move(src, dst)
    except Exception as e:
        print(f"⚠️ File move error: {e}")

    # Close tab
    if len(driver.window_handles) > 1:
        driver.close()
        driver.switch_to.window(driver.window_handles[0])
    time.sleep(1)

    # Record in summary
    summary.append({
        "TDR": tdr,
        "BOQ Files": ", ".join(boq_files) if boq_files else "Not Found",
        "NIT Files": ", ".join(nit_files) if nit_files else "Not Found"
    })

# Export summary
summary_path = os.path.join(base_download_dir, "summary.xlsx")
pd.DataFrame(summary).to_excel(summary_path, index=False)
print(f"\n✅ Completed. Summary saved to {summary_path}")
driver.quit()



Error sending stats to Plausible: error sending request for url (https://plausible.io/api/event)


🔐 Log in manually, then press ENTER to continue... 



🔎 Searching for TDR: 49452801

🔎 Searching for TDR: 49452690

🔎 Searching for TDR: 49449548
📥 BOQ via Selenium: BOQ_504401.xls

🔎 Searching for TDR: 49449515

🔎 Searching for TDR: 49449455
📥 BOQ via Selenium: BOQ_504369.xls

🔎 Searching for TDR: 49449365
⏭️ Skipping 49449365: already downloaded.

🔎 Searching for TDR: 49449364
⏭️ Skipping 49449364: already downloaded.

🔎 Searching for TDR: 49449246
⏭️ Skipping 49449246: already downloaded.

🔎 Searching for TDR: 49449245
📥 BOQ via Selenium: BOQ_504311.xls

🔎 Searching for TDR: 49449243
📥 BOQ via Selenium: BOQ_504301.xls

🔎 Searching for TDR: 49448646

🔎 Searching for TDR: 49446108
📥 BOQ via Selenium: BOQ_504167.xls

🔎 Searching for TDR: 49446088
⏭️ Skipping 49446088: already downloaded.

🔎 Searching for TDR: 49446013
⏭️ Skipping 49446013: already downloaded.

🔎 Searching for TDR: 49445492
⏭️ Skipping 49445492: already downloaded.

🔎 Searching for TDR: 49445477
⏭️ Skipping 49445477: already downloaded.

🔎 Searching for TDR: 49445409
⏭️ S

In [4]:
import os
import pandas as pd

# Load all TDRs from Excel
df = pd.read_excel("MP_pipe_data.xlsx")
tdr_list = df["TDR"].astype(str).tolist()

# Define base directory where BOQ files were downloaded
base_download_dir = os.path.join(os.getcwd(), "BOQ_Downloads")

summary = []

for tdr in tdr_list:
    tdr_dir = os.path.join(base_download_dir, tdr)
    boq_files = []
    nit_files = []
    status = ""

    if os.path.exists(tdr_dir):
        files = os.listdir(tdr_dir)
        boq_files = [f for f in files if f.lower().endswith(".xls")]
        nit_files = [f for f in files if "nit" in f.lower() and f.lower().endswith(".pdf")]
        status = "Downloaded" if boq_files or nit_files else "Empty Folder"
    else:
        status = "Not Downloaded"

    summary.append({
        "TDR": tdr,
        "Status": status,
        "BOQ Files": ", ".join(boq_files) if boq_files else "Not Found",
        "NIT Files": ", ".join(nit_files) if nit_files else "Not Found"
    })

# Save full summary
summary_path = os.path.join(base_download_dir, "summary_all.xlsx")
pd.DataFrame(summary).to_excel(summary_path, index=False)
print(f"✅ Full summary saved: {summary_path}")


✅ Full summary saved: C:\Users\Jaydeb\BOQ_Downloads\summary_all.xlsx
